<a href="https://colab.research.google.com/github/venk-meg/STRIKE/blob/main/RandomForest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Data Cleaning and Preparation

In [ ]:
# Clone repo if not already cloned
!git clone https://github.com/venk-meg/STRIKE.git
%cd STRIKE

import os
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Load all gesture CSV files
base_dir = "data_ml"
gestures = [g for g in os.listdir(base_dir) if not g.startswith(".")]

all_samples = []
for gesture in gestures:
    gesture_dir = os.path.join(base_dir, gesture)
    files = [f for f in os.listdir(gesture_dir) if f.endswith(".csv") and f != ".gitkeep"]
    for filename in files:
        df = pd.read_csv(os.path.join(gesture_dir, filename))
        df['gesture'] = gesture
        df['take_id'] = f"{gesture}_{filename.split('.')[0]}"
        all_samples.append(df)

full_df = pd.concat(all_samples, ignore_index=True)

# Cleanup: remove non-numeric or duplicate header rows
meta_cols = ['timestamp', 'gesture', 'take_id']
sensor_cols = [col for col in full_df.columns if col not in meta_cols]

for col in sensor_cols:
    full_df = full_df[full_df[col] != col]

full_df[sensor_cols] = full_df[sensor_cols].apply(pd.to_numeric, errors='coerce')
full_df[sensor_cols] = full_df[sensor_cols].ffill().bfill().fillna(0)

# Recompute sensor columns (in case of dtype coercion)
sensor_cols = full_df.select_dtypes(include='number').columns.tolist()

# Feature extraction: diff + stats per take
feature_rows = []
for take_id, group in full_df.groupby("take_id"):
    gesture = group['gesture'].iloc[0]
    diff_df = group[sensor_cols].diff().dropna()
    stats = diff_df.agg(['mean', 'std', 'min', 'max'])

    try:
        stats.columns = [f"{col}_{stat}" for col, stat in stats.columns]
    except ValueError:
        stats.columns = [f"{col}_{stats.index[0]}" for col in stats.columns]

    feature_row = stats.values.flatten()
    feature_rows.append({
        'take_id': take_id,
        'gesture': gesture,
        'features': feature_row
    })

Cloning into 'STRIKE'...
remote: Enumerating objects: 686, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 686 (delta 32), reused 24 (delta 18), pack-reused 637 (from 3)
Receiving objects: 100% (686/686), 10.01 MiB | 10.77 MiB/s, done.
Resolving deltas: 100% (280/280), done.
/content/STRIKE


## Random Forest

In [ ]:
# 📦 Install dependencies
!pip install micromlgen

# 🔢 Prepare feature matrix and labels
X = np.array([row['features'] for row in feature_rows])
y = np.array([row['gesture'] for row in feature_rows])

# 🔠 Encode labels
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 📊 Train/test split
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)

# 🌲 Train random forest on training set
from sklearn.ensemble import RandomForestClassifier
clf = RandomForestClassifier(n_estimators=5, max_depth=4, random_state=42)
clf.fit(X_train, y_train)

# 🎯 Evaluate on test set
y_pred = clf.predict(X_test)

print("✅ Accuracy:", accuracy_score(y_test, y_pred))
print("\n📊 Classification Report:\n", classification_report(y_test, y_pred, target_names=le.classes_))
print("\n🧾 Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# 🔁 Retrain on full dataset for deployment
clf.fit(X, y_encoded)

# 🧠 Export model to C using microMLgen
from micromlgen import port
class_map = {i: label for i, label in enumerate(le.classes_)}
model_code = port(clf, classmap=class_map)

# 💾 Save model.h
with open("model.h", "w") as f:
    f.write(model_code)

# 📥 Download model.h
from google.colab import files
files.download("model.h")

print(port(clf))


#pragma once
#include <cstdarg>
namespace Eloquent {
    namespace ML {
        namespace Port {
            class RandomForest {
                public:
                    /**
                    * Predict class for features vector
                    */
                    int predict(float *x) {
                        uint8_t votes[8] = { 0 };
                        // tree #1
                        if (x[45] <= -3.8299999237060547) {
                            if (x[29] <= 1.3990145325660706) {
                                if (x[54] <= -0.699999988079071) {
                                    if (x[51] <= -10.304999828338623) {
                                        votes[6] += 1;
                                    }

                                    else {
                                        votes[1] += 1;
                                    }
                                }

                                else {
                                    votes[0] += 

In [ ]:
# Installing dependies
!pip install micromlgen

# Convert to NumPy arrays
X = np.array([row['features'] for row in feature_rows])
y = np.array([row['gesture'] for row in feature_rows])

# Label encode
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Train random forest
from sklearn.ensemble import RandomForestClassifier
clf = RandomForestClassifier(n_estimators=5, max_depth=4, random_state=42)
clf.fit(X, y_encoded)

# Print label mapping
print("Label map:", dict(zip(le.classes_, le.transform(le.classes_))))

# Export to C using microMLgen
from micromlgen import port

# Create class map as dict
class_map = {i: label for i, label in enumerate(le.classes_)}

# Export to C++
model_code = port(clf, classmap=class_map)

# Save as header
with open("model.h", "w") as f:
    f.write(model_code)

from google.colab import files
files.download("model.h")


Label map: {np.str_('deleterecording'): np.int64(0), np.str_('listen'): np.int64(1), np.str_('makefist'): np.int64(2), np.str_('nextrecording'): np.int64(3), np.str_('releasefist'): np.int64(4), np.str_('rest'): np.int64(5), np.str_('selectend'): np.int64(6), np.str_('slice'): np.int64(7)}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## model evaluation

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Split your dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)

# Train on training set
clf.fit(X_train, y_train)

# Predict on test set
y_pred = clf.predict(X_test)

# Evaluate
print("✅ Accuracy:", accuracy_score(y_test, y_pred))
print("\n📊 Classification Report:\n", classification_report(y_test, y_pred, target_names=le.classes_))
print("\n🧾 Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


✅ Accuracy: 0.74

📊 Classification Report:
                  precision    recall  f1-score   support

deleterecording       0.67      1.00      0.80         6
         listen       1.00      0.50      0.67         6
       makefist       0.43      1.00      0.60         6
  nextrecording       0.80      0.67      0.73         6
    releasefist       1.00      0.67      0.80         6
           rest       0.80      0.50      0.62         8
      selectend       1.00      0.83      0.91         6
          slice       1.00      0.83      0.91         6

       accuracy                           0.74        50
      macro avg       0.84      0.75      0.75        50
   weighted avg       0.84      0.74      0.75        50


🧾 Confusion Matrix:
 [[6 0 0 0 0 0 0 0]
 [1 3 1 0 0 1 0 0]
 [0 0 6 0 0 0 0 0]
 [0 0 2 4 0 0 0 0]
 [0 0 2 0 4 0 0 0]
 [0 0 3 1 0 4 0 0]
 [1 0 0 0 0 0 5 0]
 [1 0 0 0 0 0 0 5]]
